# 📈 Machine Learning Algorithmic Trading Model Notebook
### End-to-End Pipeline: Data Ingestion, Technical Feature Engineering, Target Horizon Setup, Model Training (XGBoost & Scikit-Learn), Backtesting, and Evaluation.

This notebook provides a complete environment for building and backtesting machine learning trading strategies for any stock ticker (e.g., `TSLA`, `AAPL`, `NVDA`).
It implements:
- Dynamic Stock Ticker Selection
- Technical Indicators (SMA, RSI, Bollinger Bands, MACD, Volatility, Momentum)
- Multiclass Target Definition (Buy / Hold / Sell based on future return horizon)
- Scikit-Learn Classifiers & **XGBoost Integration with LabelEncoder**
- Strategy Backtesting Engine with Transaction Cost Simulation
- Model Evaluation (Confusion Matrix, Precision/Recall/F1, Feature Importance)


In [ ]:
# =========================================================
# 1. Setup & Imports
# =========================================================
import datetime
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Set Plotting Style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('✅ Imports and dependencies loaded successfully.')


In [ ]:
# =========================================================
# 2. Global Configuration & Strategy Parameters
# =========================================================
TICKER = 'TSLA'                       # Stock Ticker Symbol
START_DATE = '2020-01-01'              # Start Date
END_DATE = datetime.date.today().strftime('%Y-%m-%d') # End Date

# Target Definition Parameters
FUTURE_PERIOD = 30                    # Look-Ahead Horizon in Trading Days
BUY_THRESHOLD = 0.06                  # Return > +6% triggers Buy (1)
SELL_THRESHOLD = -0.06                # Return < -6% triggers Sell (-1)
ALLOW_SHORTING = True                 # Allow Shorting (-1 Position)

# Backtest Parameters
TEST_SIZE = 0.2                       # 20% Train/Test Split Ratio
INITIAL_CAPITAL = 100000              # Starting Portfolio Value ($)
COMMISSION = 0.001                    # 0.1% Transaction Fee per trade

print(f'⚙️ Configured for {TICKER} ({START_DATE} to {END_DATE})')


## 2. Data Download & Technical Feature Engineering
Fetch price data using `yfinance` and compute technical indicators used as features for the machine learning models.


In [ ]:
def load_and_engineer_data(symbol, start, end, horizon, buy_thresh, sell_thresh):
    print(f'📥 Downloading data for {symbol}...')
    raw_df = yf.download(symbol, start=start, end=end)
    if isinstance(raw_df.columns, pd.MultiIndex):
        raw_df.columns = raw_df.columns.get_level_values(0)
    raw_df.dropna(inplace=True)
    
    data = raw_df.copy()
    
    # 1. Returns & Moving Averages
    data['Return'] = data['Close'].pct_change()
    data['SMA_Short'] = data['Close'].rolling(window=20).mean()
    data['SMA_Long'] = data['Close'].rolling(window=50).mean()
    data['SMA_Ratio'] = data['SMA_Short'] / data['SMA_Long']
    
    # 2. RSI (14)
    delta = data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / (loss + 1e-9)
    data['RSI'] = 100 - (100 / (1 + rs))
    
    # 3. Bollinger Bands
    data['BB_Middle'] = data['Close'].rolling(window=20).mean()
    data['BB_Std'] = data['Close'].rolling(window=20).std()
    data['BB_Upper'] = data['BB_Middle'] + (2 * data['BB_Std'])
    data['BB_Lower'] = data['BB_Middle'] - (2 * data['BB_Std'])
    data['BB_Width'] = (data['BB_Upper'] - data['BB_Lower']) / data['BB_Middle']
    
    # 4. MACD
    ema12 = data['Close'].ewm(span=12, adjust=False).mean()
    ema26 = data['Close'].ewm(span=26, adjust=False).mean()
    data['MACD'] = ema12 - ema26
    data['MACD_Signal'] = data['MACD'].ewm(span=9, adjust=False).mean()
    
    # 5. Volatility & Momentum
    data['Volatility_10D'] = data['Return'].rolling(window=10).std()
    data['Mom_5D'] = data['Close'].pct_change(5)
    
    # 6. Future Return Target Definition (-1: Sell, 0: Neutral, 1: Buy)
    data['Future_Return'] = (data['Close'].shift(-horizon) - data['Close']) / data['Close']
    conditions = [
        data['Future_Return'] > buy_thresh,
        data['Future_Return'] < sell_thresh
    ]
    choices = [1, -1]
    data['Target'] = np.select(conditions, choices, default=0)
    
    data.dropna(inplace=True)
    return raw_df, data

raw_df, df_feat = load_and_engineer_data(
    TICKER, START_DATE, END_DATE, FUTURE_PERIOD, BUY_THRESHOLD, SELL_THRESHOLD
)
print(f'✅ Processed {len(df_feat)} rows of feature data.')


## 3. Sequential Train/Test Split & Label Encoding
Because stock data is a time series, we split sequentially (without shuffling).
We also apply `LabelEncoder` to translate target labels `[-1, 0, 1]` to `[0, 1, 2]` to enable compatibility with XGBoost.


In [ ]:
feature_cols = ['Return', 'SMA_Ratio', 'RSI', 'BB_Width', 'MACD', 'MACD_Signal', 'Volatility_10D', 'Mom_5D']
X = df_feat[feature_cols]
y = df_feat['Target']

# Sequential Time-Series Split
split_idx = int(len(df_feat) * (1 - TEST_SIZE))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
df_test = df_feat.iloc[split_idx:].copy()

# Label Encoding for XGBoost Compatibility
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print(f'Train samples: {len(X_train)} | Test samples: {len(X_test)}')
print(f'Target Label Mapping: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}')


## 4. Model Training & Comparison
Train Random Forest, Gradient Boosting, **XGBoost**, Logistic Regression, and SVM on the training set, then evaluate performance on the unseen test set.


In [ ]:
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42, eval_metric='mlogloss'),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Support Vector Machine': SVC(kernel='rbf', C=1.0)
}

results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train_encoded)
    fitted_models[name] = model
    preds_encoded = model.predict(X_test)
    preds = label_encoder.inverse_transform(preds_encoded)
    
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average='weighted', zero_division=0)
    rec = recall_score(y_test, preds, average='weighted', zero_division=0)
    f1 = f1_score(y_test, preds, average='weighted', zero_division=0)
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1
    })

results_df = pd.DataFrame(results).sort_values('F1 Score', ascending=False)
print(f'=== {TICKER} Machine Learning Model Comparison ===')
results_df


## 5. Visual Evaluation: Confusion Matrix & Feature Importances


In [ ]:
best_model_name = 'XGBoost'
best_model = fitted_models[best_model_name]
preds_encoded = best_model.predict(X_test)
preds = label_encoder.inverse_transform(preds_encoded)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, preds, labels=[-1, 0, 1])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Sell (-1)', 'Neutral (0)', 'Buy (1)'],
            yticklabels=['Sell (-1)', 'Neutral (0)', 'Buy (1)'])
axes[0].set_title(f'Confusion Matrix: {best_model_name}')
axes[0].set_xlabel('Predicted Label')
axes[0].set_ylabel('True Label')

# 2. Feature Importances
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feat_imp = pd.Series(importances, index=feature_cols).sort_values()
    feat_imp.plot(kind='barh', color='#0066CC', ax=axes[1])
    axes[1].set_title(f'Feature Importances: {best_model_name}')
    axes[1].set_xlabel('Importance Score')
else:
    axes[1].text(0.5, 0.5, 'Feature Importance unavailable for this model type', ha='center')

plt.tight_layout()
plt.show()


## 6. Algorithmic Strategy Backtesting
Simulate returns for the XGBoost model strategy vs. a Buy & Hold benchmark on unseen test data.


In [ ]:
df_test['Signal'] = preds
if ALLOW_SHORTING:
    df_test['Position'] = df_test['Signal']
else:
    df_test['Position'] = np.where(df_test['Signal'] == 1, 1, 0)

# Calculate Daily Strategy Returns including transaction fees
df_test['Strategy_Return'] = (df_test['Position'].shift(1) * df_test['Return']) - (np.abs(df_test['Position'].diff()) * COMMISSION)
df_test.dropna(inplace=True)

# Cumulative Growth
df_test['Cum_Benchmark'] = (1 + df_test['Return']).cumprod() * INITIAL_CAPITAL
df_test['Cum_Strategy'] = (1 + df_test['Strategy_Return']).cumprod() * INITIAL_CAPITAL

# Plot Backtest Results
plt.figure(figsize=(12, 6))
plt.plot(df_test.index, df_test['Cum_Strategy'], label=f'{best_model_name} ML Strategy', color='#00CC66', linewidth=2)
plt.plot(df_test.index, df_test['Cum_Benchmark'], label=f'Buy & Hold {TICKER}', color='#E82127', linestyle='--', linewidth=1.5)
plt.title(f'{TICKER} Algorithmic Trading Strategy Backtest ({df_test.index[0].strftime("%Y-%m-%d")} to {df_test.index[-1].strftime("%Y-%m-%d")})')
plt.ylabel('Portfolio Value ($)')
plt.xlabel('Date')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Performance Metrics
strat_ret_pct = ((df_test['Cum_Strategy'].iloc[-1] - INITIAL_CAPITAL) / INITIAL_CAPITAL) * 100
bench_ret_pct = ((df_test['Cum_Benchmark'].iloc[-1] - INITIAL_CAPITAL) / INITIAL_CAPITAL) * 100
daily_strat = df_test['Strategy_Return']
sharpe = (daily_strat.mean() / (daily_strat.std() + 1e-9)) * np.sqrt(252)
cum_max = df_test['Cum_Strategy'].cummax()
drawdown = (df_test['Cum_Strategy'] - cum_max) / cum_max
max_drawdown = drawdown.min() * 100

print(f'=== Backtest Performance Summary ({TICKER}) ===')
print(f'ML Strategy Return:  {strat_ret_pct:+.2f}%')
print(f'Buy & Hold Return:   {bench_ret_pct:+.2f}%')
print(f'Annualized Sharpe:   {sharpe:.2f}')
print(f'Max Drawdown:        {max_drawdown:.2f}%')
print(f'Total Trades:        {int(np.abs(df_test["Position"].diff()).sum())}')


## 7. Live Prediction Sandbox
Use this function to test custom technical indicator inputs and inspect real-time Buy/Sell/Hold predictions from the trained model.


In [ ]:
def predict_live_signal(model, encoder, return_val, sma_ratio, rsi, bb_width, macd, macd_sig, vol_10d, mom_5d):
    sample_df = pd.DataFrame([{
        'Return': return_val,
        'SMA_Ratio': sma_ratio,
        'RSI': rsi,
        'BB_Width': bb_width,
        'MACD': macd,
        'MACD_Signal': macd_sig,
        'Volatility_10D': vol_10d,
        'Mom_5D': mom_5d
    }])
    pred_encoded = model.predict(sample_df)
    pred_signal = encoder.inverse_transform(pred_encoded)[0]
    
    if pred_signal == 1:
        signal_str = f'🟢 BUY SIGNAL (1) - Expecting future return > +{BUY_THRESHOLD*100:.1f}%'
    elif pred_signal == -1:
        signal_str = f'🔴 SELL / SHORT SIGNAL (-1) - Expecting future return < {SELL_THRESHOLD*100:.1f}%'
    else:
        signal_str = f'🟡 NEUTRAL / HOLD SIGNAL (0) - Expecting sideways return'
        
    return signal_str

# Example Test Prediction
test_signal = predict_live_signal(
    best_model, label_encoder,
    return_val=0.015, sma_ratio=1.03, rsi=62.0, bb_width=0.045, macd=0.8, macd_sig=0.3, vol_10d=0.02, mom_5d=0.03
)
print('Prediction Output:')
print(test_signal)
